<a href="https://colab.research.google.com/github/fareehaikram93/assignment1-intern/blob/main/work/notebooks/w04_baseline_score(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Load the contracted dataset

*Not one of the four templated sections below — added because the rule needs data to run against. Uses the same contracted dataset and manifest from the data-contract notebook (`ranking_signal_content_grain_*.csv` + `.manifest.json`), so this baseline is built on the exact same grain and population as everything downstream.*

In [4]:
import json
import pandas as pd

# Point these at the files written by ranking_signal_data_contract.ipynb
DATASET_PATH = '/content/ranking_signal_content_grain_20260811T100648Z.csv'
MANIFEST_PATH ='/content/ranking_signal_content_grain_20260811T100648Z.manifest.json'

df = pd.read_csv(DATASET_PATH)
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

print(f"Rows: {len(df):,}")
print(f"Contract version: {manifest['contract_version']}, grain: {manifest['grain']}")

# Grain guard — the classic mistake this skill warns about. Must print 0 rows.
pk_cols = manifest['grain']
dupes = df.duplicated(subset=pk_cols).sum()
assert dupes == 0, f"{dupes} duplicate grain keys — fix the join upstream before scoring anything"
print(f"Grain check passed: 0 duplicate ({pk_cols[0]}, {pk_cols[1]}) pairs.")

df.head()

Rows: 100,000
Contract version: 1.0.0, grain: ['client_hash_id', 'content_hash_id']
Grain check passed: 0 duplicate (client_hash_id, content_hash_id) pairs.


,client_hash_id,content_hash_id,gsc_avg_position_90d,days_observed,search_volume,competition,competition_level,cpc,char_count,main_intent,...,keyword_token_count,url_char_count,backlinks,category_count,word_count,content_type,provider_used,model_used,content_created_date,content_updated_date
0,client_73cda7b4e4f265ea,content_ef89ff748f7faed7,8.541545,91,0.0,0.00,LOW,0.00,19397.0,informational,...,5,83,0.0,0,2883.0,keyword article,google,gemini-3-flash-preview,2026-02-10,2026-05-20
1,client_73cda7b4e4f265ea,content_36166ace2994c2d6,17.373375,64,90.0,0.86,HIGH,0.27,NaN,transactional,...,6,91,NaN,0,NaN,keyword article,NaN,NaN,2025-03-03,2026-05-18
2,client_e547b89c05043229,content_f6963771704e02f9,6.866495,91,10.0,0.02,LOW,0.00,17435.0,informational,...,8,114,NaN,0,2715.0,keyword article,NaN,gemini-3-flash-preview,2025-02-07,2026-07-06
3,client_73cda7b4e4f265ea,content_efe1c906f4d7d66d,3.219822,91,40.0,0.01,LOW,0.00,NaN,informational,...,6,108,NaN,0,NaN,keyword article,NaN,NaN,2025-02-14,2026-05-18
4,client_73cda7b4e4f265ea,content_4d5bc13d6cfd81a6,6.780990,91,20.0,0.00,LOW,0.00,NaN,transactional,...,6,127,NaN,0,NaN,keyword article,NaN,NaN,2025-07-31,2026-07-01


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in one sentence:** a page is worth reviewing first if it has real search demand
but a weak ranking position — because that combination is where an editor's time is most likely
to pay off, more than a page nobody is searching for or a page that's already ranking well.

Three supporting signals make the pick more or less confident, but none of them gate the score
on their own:
- **stale** — hasn't been updated in a long time (freshness signal, not a guess about content quality)
- **thin** — short relative to what usually ranks in this dataset (an editable, checkable fact)
- **low-competition opportunity** — real demand exists but the keyword space isn't crowded, so effort
  is more likely to move the needle

**Reason codes this rule can output** (a row can carry more than one):

| Reason code | Fires when |
|---|---|
| `weak_position_with_demand` | `search_volume >= 10` **and** `gsc_avg_position_90d > 10` (not on page one) — this is the gate; nothing scores above zero without it |
| `stale_content` | `days_since_update >= 180` |
| `thin_content` | `word_count < 1200` |
| `low_competition_opportunity` | `competition_level == 'LOW'` **and** `search_volume >= 10` |
| `general_review_candidate` | none of the above fired, but the row is in the dataset (fallback label only — never scores above zero) |

Everything here is knowable *before* today — none of it is a future outcome, and none of it is a
product-computed decision flag (this dataset doesn't ship any).

In [5]:
# --- Rule parameters — plain thresholds, no fitted weights ---
MIN_SEARCH_VOLUME = 10          # minimum monthly searches to call it "real demand"
WEAK_POSITION_THRESHOLD = 10    # positions worse than this are off page one
STALE_DAYS_THRESHOLD = 180      # ~6 months since last update
THIN_WORD_COUNT = 1200          # below this, word count itself looks incomplete

# content_updated_date is a raw date in the contracted dataset; derive an age-in-days
# feature the same way the modeling notebook does, so the two notebooks agree with each other.
SNAPSHOT_DATE = pd.Timestamp(manifest['snapshot_date'])
df['content_updated_date'] = pd.to_datetime(df['content_updated_date'], errors='coerce')
df['days_since_update'] = (SNAPSHOT_DATE - df['content_updated_date']).dt.days

def reason_codes(row) -> list[str]:
    reasons = []
    if row['search_volume'] >= MIN_SEARCH_VOLUME and row['gsc_avg_position_90d'] > WEAK_POSITION_THRESHOLD:
        reasons.append('weak_position_with_demand')
    if pd.notna(row['days_since_update']) and row['days_since_update'] >= STALE_DAYS_THRESHOLD:
        reasons.append('stale_content')
    if pd.notna(row['word_count']) and row['word_count'] < THIN_WORD_COUNT:
        reasons.append('thin_content')
    if row.get('competition_level') == 'LOW' and row['search_volume'] >= MIN_SEARCH_VOLUME:
        reasons.append('low_competition_opportunity')
    if not reasons:
        reasons.append('general_review_candidate')
    return reasons

def suggested_action(reasons: list[str]) -> str:
    if 'thin_content' in reasons:
        return 'expand_and_refresh'
    if 'stale_content' in reasons:
        return 'refresh_content'
    if 'low_competition_opportunity' in reasons:
        return 'quick_win_review'
    if 'weak_position_with_demand' in reasons:
        return 'review_position_opportunity'
    return 'monitor'

def confidence_label(reasons: list[str]) -> str:
    # Confidence rises with how many SUPPORTING signals back up the primary gate —
    # not just whether the gate fired at all.
    supporting = {'stale_content', 'thin_content', 'low_competition_opportunity'}
    support_count = len(supporting.intersection(reasons))
    if 'weak_position_with_demand' not in reasons:
        return 'low'
    if support_count >= 2:
        return 'high'
    if support_count == 1:
        return 'medium'
    return 'low'

print('Rule, reason codes, action mapping, and confidence logic defined.')

Rule, reason codes, action mapping, and confidence logic defined.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import numpy as np

# --- The score itself: transparent, multiplicative, readable in one line ---
# Zero unless the gate (demand + weak position) fires. Scaled by search_volume (bigger
# audience = bigger opportunity) and by how far past the page-one threshold the position sits
# (a page at position 40 is a bigger gap than a page at position 11).
has_demand = (df['search_volume'].fillna(0) >= MIN_SEARCH_VOLUME).astype(int)
weak_position = (df['gsc_avg_position_90d'] > WEAK_POSITION_THRESHOLD).astype(int)
position_gap = (df['gsc_avg_position_90d'] - WEAK_POSITION_THRESHOLD).clip(lower=0)

df['action_score'] = (
    weak_position * has_demand * df['search_volume'].fillna(0) * (1 + position_gap / 100)
)

df['reason_code_list'] = df.apply(reason_codes, axis=1)
df['reason_codes'] = df['reason_code_list'].apply(lambda r: '|'.join(r))
df['suggested_action'] = df['reason_code_list'].apply(suggested_action)
df['confidence'] = df['reason_code_list'].apply(confidence_label)

df['baseline_rank'] = df['action_score'].rank(method='first', ascending=False).astype(int)
ranked = df.sort_values('baseline_rank')

qualifying = (df['action_score'] > 0).sum()
print(f"Rows scored: {len(df):,}")
print(f"Rows that cleared the gate (score > 0): {qualifying:,} ({qualifying / len(df):.1%})")
print(f"Top score: {ranked['action_score'].max():.1f}   Median (qualifying only): "
      f"{ranked.loc[ranked['action_score'] > 0, 'action_score'].median():.1f}")

output_columns = [
    'baseline_rank', 'client_hash_id', 'content_hash_id', 'action_score',
    'confidence', 'suggested_action', 'reason_codes',
    'gsc_avg_position_90d', 'search_volume', 'competition_level',
    'word_count', 'days_since_update', 'days_observed',
]
output_columns = [c for c in output_columns if c in ranked.columns]

import os
os.makedirs('work/outputs', exist_ok=True)
OUTPUT_PATH = 'work/outputs/baseline_action_score.csv'
ranked[output_columns].to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {OUTPUT_PATH}")

Rows scored: 100,000
Rows that cleared the gate (score > 0): 38,551 (38.6%)
Top score: 349136.2   Median (qualifying only): 21.9
Wrote work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top20 = ranked.head(20)[output_columns].reset_index(drop=True)
top20.index = top20.index + 1
top20

,baseline_rank,client_hash_id,content_hash_id,action_score,confidence,suggested_action,reason_codes,gsc_avg_position_90d,search_volume,competition_level,word_count,days_since_update,days_observed
1,1,client_fef1a8f436438636,content_ac0c525eb243379b,349136.211602,medium,quick_win_review,weak_position_with_demand|low_competition_oppo...,51.925289,246000.0,LOW,NaN,13,91
2,2,client_fef1a8f436438636,content_0184167e6037fbc7,313149.281379,medium,quick_win_review,weak_position_with_demand|low_competition_oppo...,65.795662,201000.0,LOW,NaN,13,87
3,3,client_fef1a8f436438636,content_a31c400b511b1458,245655.616617,medium,quick_win_review,weak_position_with_demand|low_competition_oppo...,32.216725,201000.0,LOW,NaN,13,50
4,4,client_fef1a8f436438636,content_7e6779733b1dd409,237629.075795,medium,quick_win_review,weak_position_with_demand|low_competition_oppo...,54.017622,165000.0,LOW,NaN,13,91
5,5,client_9d54435aabd95a6c,content_28a33398953711ce,205375.937500,medium,quick_win_review,weak_position_with_demand|low_competition_oppo...,12.177083,201000.0,LOW,2991.0,22,32
6,6,client_fef1a8f436438636,content_ff42f4a65f10744c,204726.055118,medium,quick_win_review,weak_position_with_demand|low_competition_oppo...,61.648930,135000.0,LOW,NaN,13,89
7,7,client_fef1a8f436438636,content_d65ca6a7e33f6659,161274.281997,high,expand_and_refresh,weak_position_with_demand|thin_content|low_com...,56.612984,110000.0,LOW,1074.0,-1,91
8,8,client_fef1a8f436438636,content_9755ef5214465568,149245.725394,medium,quick_win_review,weak_position_with_demand|low_competition_oppo...,45.677932,110000.0,LOW,NaN,13,68
9,9,client_fef1a8f436438636,content_4a779911b15a99f5,139179.556235,medium,quick_win_review,weak_position_with_demand|low_competition_oppo...,63.789565,90500.0,LOW,NaN,13,91
10,10,client_fef1a8f436438636,content_e7e56b5396c93880,138099.846817,medium,quick_win_review,weak_position_with_demand|low_competition_oppo...,35.545315,110000.0,LOW,2912.0,13,91


**Read the printed table above, row by row, before writing anything below.** The columns
already give you the action, the reason code(s), and a confidence label computed from how many
supporting signals fired — that part is mechanical. What the table *can't* do for you is the part
that actually needs a human: does the reason make sense for *this specific row*, and what
observation would make you distrust the pick?

For each of the 20 rows, write one line in the form:

> **Rank N** — `action` / `reason_codes` — *confidence note*: why the confidence label feels right
> or not — *would be wrong if*: the one fact about this row that would change your mind.

Example of the level of specificity expected (not a real row — replace with your own):

> **Rank 1** — `review_position_opportunity` / `weak_position_with_demand|stale_content` —
> *confidence note*: medium confidence is fair here, only one supporting signal beyond the gate —
> *would be wrong if*: `days_observed` turns out to be very low (few days of position data), in
> which case the 90-day average position is noisy, not a real signal.

Fill in your own 20 lines here:

1.
2.
3.
4.
5.
6.
7.
8.
9.
10.
11.
12.
13.
14.
15.
16.
17.
18.
19.
20.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# --- Automated weak-pick screen: flags patterns worth a second look, doesn't decide for you ---
weak_pick_flags = []

# 1. Very young content flagged as a "weak position" opportunity may just need more time,
#    not a content fix — content_age_days isn't in this table directly, so use days_observed
#    as a proxy: very few observed days means the 90-day average is thin.
thin_history = top20[top20.get('days_observed', pd.Series(dtype=float)) < 30] if 'days_observed' in top20.columns else pd.DataFrame()
if len(thin_history):
    weak_pick_flags.append(
        f"{len(thin_history)} of the top 20 have fewer than 30 days of observed position data "
        "(days_observed < 30) — their 90-day average position rests on a short window and may "
        "be noisier than the score treats it."
    )

# 2. Rows scoring purely on a single reason code (no supporting signal) are the ones a human
#    should sanity-check hardest — they only cleared the gate, nothing backs it up further.
low_confidence_in_top20 = (top20['confidence'] == 'low').sum() if 'confidence' in top20.columns else 0
if low_confidence_in_top20:
    weak_pick_flags.append(
        f"{low_confidence_in_top20} of the top 20 are 'low' confidence — they cleared the gate "
        "on demand + position alone, with no supporting reason code. Worth a closer manual look."
    )

if weak_pick_flags:
    print("Automated flags for manual review:")
    for flag in weak_pick_flags:
        print(f" - {flag}")
else:
    print("No automated weak-pick flags fired on the top 20 — still read every row by hand above; "
          "this check catches known failure patterns, not everything.")

No automated weak-pick flags fired on the top 20 — still read every row by hand above; this check catches known failure patterns, not everything.


**Leakage checklist, run out loud against this rule:**

- [x] **No product decision flags used.** This dataset never shipped `health_score`,
  `priority_score`, `action_type`, or any other FlyRank product output — there was nothing to
  accidentally include.
- [x] **No future window used.** Every field in the rule (`search_volume`, `gsc_avg_position_90d`,
  `word_count`, `content_updated_date`) is knowable as of the frozen snapshot date
  recorded in `manifest["snapshot_date"]` — nothing here reaches past it.
- [x] **Using `gsc_avg_position_90d` here is not label leakage.** This notebook doesn't train a
  model — it's a transparent, hand-written rule that ranks pages by an *already-observed*
  outcome, exactly the way `trend_direction` is used directly in the internship's other lane
  baseline. The distinction that matters: this field must never be fed as a *feature* into a
  model that predicts it (the ML pipeline notebook already excludes it correctly, since it's the
  regression target there, not an input).
- [x] **IDs used for grouping only.** `client_hash_id` / `content_hash_id` appear in the output
  for traceability, never as inputs to the score itself.
- [ ] **Client-level skew.** Not yet checked — worth confirming the top 20 isn't dominated by a
  single client before treating this as a general finding. One line to check:
  `ranked.head(50)['client_hash_id'].value_counts()`.

**Public-safety check:** the output CSV and this notebook contain only pseudonymized IDs,
aggregated metrics, and reason codes — no client names, domains, URLs, or raw queries.

In [9]:
# Quick client-skew check referenced in the leakage checklist above.
if 'client_hash_id' in ranked.columns:
    top50_client_counts = ranked.head(50)['client_hash_id'].value_counts()
    print("Client representation in the top 50 (top 5 shown):")
    print(top50_client_counts.head())
    top_client_share = top50_client_counts.iloc[0] / 50
    if top_client_share > 0.5:
        print(f"\n\u26a0\ufe0f  One client holds {top_client_share:.0%} of the top 50 — the queue "
              "currently reads more like 'one client's opportunities' than a general finding. Say "
              "so explicitly if you keep this version.")
    else:
        print(f"\nNo single client dominates the top 50 (largest share: {top_client_share:.0%}).")

Client representation in the top 50 (top 5 shown):
client_hash_id
client_fef1a8f436438636    34
client_73cda7b4e4f265ea     8
client_e547b89c05043229     4
client_9d54435aabd95a6c     2
client_a80fca3f171ed1de     1
Name: count, dtype: int64

⚠️  One client holds 68% of the top 50 — the queue currently reads more like 'one client's opportunities' than a general finding. Say so explicitly if you keep this version.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.